# Week 2 Day 2 assignment, task 2: a memory about each customer

The memory agent gets notes on each customer: a Markdown file for each `customer_id`, read with `read_customer_notes` and added to with `add_customer_note`, both in `week2day2_common.py`. `SEED_NOTES` there starts 7 customers with notes: three harmless (contact preferences, earlier calls) and four relevant ones that don't change the decision (a police report and an itemized bill on their way, questions about Electronics Plus and diving). Run task 1 first; its runs are the comparison.

In [ ]:
import shutil

from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from week2day2_common import (
    MEMORY_LINE, NOTES, RESULTS, add_customer_note, calculate_payout, compare, decide, keep_notes, load_runs, luna,
    make_system_prompt, memory_search_tools, read_customer_notes, record_decision, reset_notes, run_experiment,
)

## What should happen to the notes between runs?

Reset them to the starting notes before every run, as the lab clears `recorded`. Otherwise run 2 reads what run 1 wrote about the same claim, such as "approved $562.95", and:

- the answer leaks: the agent can copy its earlier decision instead of deciding;
- the runs stop being repeats of one test, so averaging them no longer measures how much the agent varies;
- a note that the claim is already decided can make the agent refer it, or record nothing.

In production the notes should carry over; that's the point of a memory. In an evaluation every run starts from the same state. The last experiment below shows what happens without the reset.

In [ ]:
NOTES_LINE = "Read the customer's notes before you decide. After you record your decision, add a short note on the customer for the next handler: what was claimed, what you decided and anything still outstanding.\n"

memory_search = await memory_search_tools()
notes_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, *memory_search, read_customer_notes, add_customer_note],
    system_prompt=make_system_prompt(MEMORY_LINE, NOTES_LINE),
    middleware=[ToolRetryMiddleware(max_retries=3)],
)


async def notes_task(*, item, **kwargs):
    return await decide(notes_agent, item.input, search_tools=["qdrant-find"])


notes_table = run_experiment("Payout tool, memory and customer notes", notes_task, "notes", before_run=reset_notes, after_run=keep_notes("notes"))
tables = {"Memory (task 1)": load_runs("qdrant"), "Memory and customer notes": notes_table}
compare(tables)

In [ ]:
notes_table.assign(
    read_notes=notes_table["tool_calls"].apply(lambda calls: "read_customer_notes" in calls),
    added_note=notes_table["tool_calls"].apply(lambda calls: "add_customer_note" in calls),
).groupby("run")[["read_notes", "added_note"]].mean()

## Without the reset

One more run, starting from the notes that run 3 left.

In [ ]:
def notes_from_run_3(run):
    shutil.rmtree(NOTES, ignore_errors=True)
    shutil.copytree(RESULTS / "notes_notes_run_3", NOTES)


kept = run_experiment("Payout tool, memory and customer notes, not reset", notes_task, "notes_kept", runs=1, before_run=notes_from_run_3).set_index("claim_id")
kept["run_3"] = notes_table[notes_table["run"] == 3].set_index("claim_id")["agent"]
print(compare({**tables, "Not reset": kept.reset_index()}))
kept[(kept["agent"] != kept["run_3"]) | kept["rationale"].str.contains("prior customer notes|already approved", case=False)][["run_3", "agent", "right", "rationale"]]

## Conclusion

| | Memory (task 1) | Memory and notes | Not reset (1 run) |
|---|---:|---:|---:|
| Right | 70 of 75 (23, 24, 23) | 74 of 75 (25, 25, 24) | 22 of 25 |
| Cost of mistakes, per run | $590 | $17 | $150 |
| Model cost, per run | $0.012 | $0.016 | $0.016 |

- The agent read the notes and added one on every claim in every run. Its notes read like a handler's: the claim, the decision and payout, and what's outstanding.
- The starting notes did no harm: the 7 claims with them were right in every run, including the two referrals waiting on a document.
- The better score is not the notes' doing. Task 1's mistakes were on claims without starting notes, and with one claim per customer a note can't reach another claim within a run. The notes add a third to the model cost.
- Without the reset, the agent referred CLM-1304 and CLM-1320 "for a duplicate-claim/payment status check", because its own notes said they were paid. On CLM-1302, unsure of the storm date, it approved because "prior customer notes record this same claim as approved for $2,473": right, but copied.

Referring a claim the notes say is already paid is what a careful agent should do in production. In an evaluation it means the agent answers a different question from the dataset's, so the notes are reset before every run. Testing whether notes help across claims needs a dataset with several claims per customer.